# 1. Preliminares

In [1]:
!pip install xgboost shap optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 10.8 MB/s eta 0:00:00


## 1.1 Importar librerías

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import requests
import json
import warnings
warnings.filterwarnings('ignore')
import sklearn
from sklearn.linear_model    import Ridge
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.metrics         import (classification_report, confusion_matrix, f1_score, roc_auc_score, ConfusionMatrixDisplay, recall_score, roc_curve, precision_recall_curve,
                                     average_precision_score, precision_score, accuracy_score)
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedGroupKFold
from sklearn.pipeline        import Pipeline
from xgboost                 import XGBClassifier
import shap
from scipy.linalg            import eigh
from libpysal.weights        import KNN
import libpysal
import joblib
import time
import geopandas as gpd
import optuna
import os
import shutil
import sys

np.random.seed(42)





## 1.2 Crear funciones

In [3]:
# ── Paleta ──
C = {
    'bajo':    '#27ae60',
    'medio':   '#e67e22',
    'alto':    '#c0392b',
    'primary': '#1a2940',
    'accent':  '#2980b9',
    'light':   '#f8f9fa',
    'mid':     '#95a5a6',
}
ORDEN  = ['bajo', 'medio', 'alto']
C_RISK = {k: C[k] for k in ORDEN}

# ── Estilo matplotlib global ──
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.edgecolor':    '#dee2e6',
    'axes.linewidth':    0.8,
    'axes.grid':         True,
    'grid.color':        '#eeeeee',
    'grid.linewidth':    0.5,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.titlesize':    12,
    'axes.titleweight':  'bold',
    'axes.titlepad':     10,
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'legend.framealpha': 0.9,
})

def anotar(ax, xs, ys, fmt='{:.0f}', color='#1a2940'):
    for x, y in zip(xs, ys):
        ax.annotate(fmt.format(y), (x, y), textcoords='offset points',
                    xytext=(0, 9), ha='center', fontsize=8.5, color=color)

def covid_line(ax):
    ax.axvline(2020, color=C['mid'], ls='--', lw=1.2, alpha=0.7)
    ax.text(2020.08, ax.get_ylim()[1]*0.93, 'COVID-19',
            fontsize=8, color=C['mid'], style='italic')


def obtener_altitud_lote(coords_list, batch_size=100):
    """
    Obtiene altitud en metros para una lista de (lat, lon).
    Usa Open Elevation API (gratuita).
    coords_list: lista de (lat, lon)
    Retorna lista de altitudes en metros.
    """
    url       = "https://api.open-elevation.com/api/v1/lookup"
    altitudes = []

    for i in range(0, len(coords_list), batch_size):
        lote = coords_list[i:i+batch_size]
        payload = {"locations": [{"latitude": lat, "longitude": lon}
                                  for lat, lon in lote]}
        try:
            resp = requests.post(url, json=payload, timeout=30)
            if resp.status_code == 200:
                resultados = resp.json()['results']
                altitudes.extend([r['elevation'] for r in resultados])
            else:
                altitudes.extend([np.nan] * len(lote))
        except Exception:
            altitudes.extend([np.nan] * len(lote))

        if i % 500 == 0 and i > 0:
            print(f"  Procesados {i}/{len(coords_list)} municipios...")
            time.sleep(1)  # respetar rate limit

    return altitudes

def clasificar_piso_termico(alt):
    if pd.isna(alt):   return np.nan
    if alt < 900:      return 0   # tierra caliente (Caribe, Pacífico, Llanos, Amazonía)
    if alt < 2000:     return 1   # tierra templada (cafetero)
    return 2                       # tierra fría (altiplano)

def haversine(lat1, lon1, lat2, lon2):
    """Distancia en km entre dos puntos geográficos."""
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi  = np.radians(lat2 - lat1)
    dlam  = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def distancia_a_capital(row):
    """Distancia en km del municipio a su capital departamental."""
    cod_dpto = row['cod_mun'][:2]
    if cod_dpto not in CAPITALES:
        return np.nan
    _, lat_cap, lon_cap = CAPITALES[cod_dpto]
    return haversine(row['latitud'], row['longitud'], lat_cap, lon_cap)

def distancia_a_bogota(row):
    """Distancia en km a Bogotá (proxy de conectividad nacional)."""
    return haversine(row['latitud'], row['longitud'], 4.711, -74.072)

def distancia_a_ciudad_mas_cercana(row):
    """Distancia al centro urbano más cercano entre todas las capitales."""
    min_dist = np.inf
    for _, (_, lat_c, lon_c) in CAPITALES.items():
        d = haversine(row['latitud'], row['longitud'], lat_c, lon_c)
        if d < min_dist:
            min_dist = d
    return min_dist



VARS_IPM = [
    'analfabetismo','bajo_logro_educativo','barreras_primera_infancia',
    'barreras_acceso_salud','tasa_dependencia','hacinamiento_critico',
    'inadec_eliminacion_excretas','inasistencia_escolar','inadec_paredes',
    'inadec_pisos','rezago_escolar','sin_agua_mejorada',
    'sin_aseguramiento_salud','trabajo_infantil','trabajo_informal'
]

ETIQ = {
    'analfabetismo':'Analfabetismo','bajo_logro_educativo':'Bajo logro educativo',
    'barreras_primera_infancia':'Barreras 1ª infancia',
    'barreras_acceso_salud':'Barreras acceso salud',
    'tasa_dependencia':'Tasa dependencia','hacinamiento_critico':'Hacinamiento crítico',
    'inadec_eliminacion_excretas':'Inadec. excretas',
    'inasistencia_escolar':'Inasistencia escolar',
    'inadec_paredes':'Paredes inadecuadas','inadec_pisos':'Pisos inadecuados',
    'rezago_escolar':'Rezago escolar','sin_agua_mejorada':'Sin agua mejorada',
    'sin_aseguramiento_salud':'Sin aseg. salud','trabajo_infantil':'Trabajo infantil',
    'trabajo_informal':'Trabajo informal',
}

## 1.3 Carga de datos

**Configuración de Rutas**

In [4]:
# Si corre en Colab, monta Drive; si no, asume que el repo está
# clonado y las rutas son relativas a la raíz del proyecto.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/Máster Data Science/TFM')
except ImportError:
    BASE_DIR = Path.cwd()

RUTA_DATOS = BASE_DIR / 'Bases de Datos'
RUTA_IMAGENES = BASE_DIR / 'outputs' / 'figuras'
RUTA_DOCKER = BASE_DIR / 'docker'
RUTA_DATOS_PROCESADOS = BASE_DIR / 'Bases de Datos' / 'procesados'
RUTA_MODELOS = BASE_DIR / 'Modelos'

# Crea las carpetas si no existen
RUTA_IMAGENES.mkdir(parents=True, exist_ok=True)
RUTA_DOCKER.mkdir(parents=True, exist_ok=True)
RUTA_DATOS_PROCESADOS.mkdir(parents=True, exist_ok=True)
RUTA_MODELOS.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


**Lectura de Datos**

In [5]:
sivigila_2016 = pd.read_excel(RUTA_DATOS / 'Datos_2016_113.xlsx')
sivigila_2017 = pd.read_excel(RUTA_DATOS / 'Datos_2017_113.xlsx')
sivigila_2018 = pd.read_excel(RUTA_DATOS / 'Datos_2018_113.xls')
sivigila_2019 = pd.read_excel(RUTA_DATOS / 'Datos_2019_113.xls')
sivigila_2020 = pd.read_excel(RUTA_DATOS / 'Datos_2020_113.xls')
sivigila_2021 = pd.read_excel(RUTA_DATOS / 'Datos_2021_113.xls')
sivigila_2022 = pd.read_excel(RUTA_DATOS / 'Datos_2022_113.xls')
sivigila_2023 = pd.read_excel(RUTA_DATOS / 'Datos_2023_113.xlsx')
sivigila_2024 = pd.read_excel(RUTA_DATOS / 'Datos_2024_113.xlsx')


ipm = pd.read_excel(
    RUTA_DATOS / 'ipm_2018.xlsx',
    sheet_name='6_Privaciones IPM Dpt-Mpio',
    header=1
)

mdm_indicadores = pd.read_csv(
    RUTA_DATOS / 'mdm_componentes.csv',
    dtype={'Código Entidad': str, 'Código Departamento': str}
)

pob_raw = pd.read_excel(
    RUTA_DATOS / 'poblacion.xlsx',
    sheet_name='PobMunicipalxÁreaSexoEdad',
    header=None,  # Leer sin header para construirlo manualmente
    skiprows=7    # saltar filas 0-6 (títulos e imágenes)
)

pob_raw_17 = pd.read_excel(RUTA_DATOS / 'pob_2017.xlsx')

**Coordenadas de los municipios**

In [6]:
url = "https://gist.githubusercontent.com/brolin/ac2f2b871f98c4d79272/raw/current.csv"
df_coord= pd.read_csv(url)

df_coord['DIVIPOLA'] = df_coord['DIVIPOLA'].astype(str).str.zfill(5)
print(df_coord.shape)
df_coord.head()

(1124, 8)


,x,y,NOMBRE,DIVIPOLA,AREA,ENTIDAD,DEPTO,CODDEPTO
0,-75.611037,8.848040,Ciénaga de Oro,23189,644,Cabecera Municipal,Córdoba,23
1,-75.408499,8.479396,Pueblo Nuevo,23570,819,Cabecera Municipal,Córdoba,23
2,-75.049999,8.260972,Ayapel,23068,1929,Cabecera Municipal,Córdoba,23
3,-75.770297,7.711339,Puerto Libertador,23580,2062,Cabecera Municipal,Córdoba,23
4,-75.910535,8.989570,San Pelayo,23686,470,Cabecera Municipal,Córdoba,23


**GeoJASON**

In [7]:
# Archivo GeoJSON del DANE
with open(RUTA_DATOS / 'municipios.geojson', 'r', encoding='utf-8') as f:
    geojson_dane = json.load(f)

## 1.4 Creación de la variable objetivo

**Preparar la base de datos de población**

In [8]:
# Las dos primeras filas son el header partido
fila_header_1 = pob_raw.iloc[0].ffill()  # rellena celdas combinadas
fila_header_2 = pob_raw.iloc[1]

# Construir nombres de columna combinando ambas filas
nombres = []
for h1, h2 in zip(fila_header_1, fila_header_2):
    h1 = str(h1).strip() if pd.notna(h1) else ''
    h2 = str(h2).strip() if pd.notna(h2) else ''
    if h2 and h2 != 'nan':
        nombres.append(h2)
    else:
        nombres.append(h1)

# Asignar nombres y eliminar las filas de header
pob_raw.columns = nombres
pob = pob_raw.iloc[2:].reset_index(drop=True)

print("Columnas:", pob.columns.tolist()[:10], '...')
print("Shape:", pob.shape)
print(pob.head(3))

Columnas: ['DP', 'DPNOM', 'MPIO', 'DPMP', 'AÑO', 'ÁREA GEOGRÁFICA', 'Total', 'Hombres', 'Mujeres', 'Hombres 0 años'] ...
Shape: (84233, 312)
   DP      DPNOM   MPIO      DPMP   AÑO                    ÁREA GEOGRÁFICA  \
0  05  Antioquia  05001  Medellín  2018                 Cabecera Municipal   
1  05  Antioquia  05001  Medellín  2018  Centros Poblados y Rural Disperso   
2  05  Antioquia  05001  Medellín  2018                              Total   

     Total  Hombres  Mujeres Hombres 0 años  ... Total 91 años Total 92 años  \
0  2382405  1118868  1263537          13512  ...          2027          1624   
1    44728    22475    22253            304  ...            25            20   
2  2427133  1141343  1285790          13816  ...          2052          1644   

  Total 93 años Total 94 años Total 95 años Total 96 años Total 97 años  \
0          1282          1001           775           599           458   
1            16            12             9             7             5   


In [9]:
# Filtrar solo área Total
pob_total = pob[pob['ÁREA GEOGRÁFICA'] == 'Total'].copy()

# Columnas de 0 a 4 años (Total ambos sexos)
cols_0_4 = [
    'Total 0 años', 'Total 1 año', 'Total 2 años',
    'Total 3 años', 'Total 4 años'
]

cols_disponibles = [c for c in cols_0_4 if c in pob_total.columns]

# Convertir a numérico y sumar
pob_total[cols_disponibles] = pob_total[cols_disponibles].apply(pd.to_numeric, errors='coerce')
pob_total['pob_0_4'] = pob_total[cols_disponibles].sum(axis=1)

# Estandarizar código DIVIPOLA
pob_total['cod_mun'] = pob_total['MPIO'].astype(str).str.zfill(5)

# Filtrar años necesarios y seleccionar columnas
pob_filtrado = pob_total[
    pob_total['AÑO'].astype(int).isin([2018, 2019, 2020, 2021, 2022, 2023, 2024])
][['cod_mun', 'AÑO', 'pob_0_4']].rename(columns={'AÑO': 'año'}).reset_index(drop=True)

print(f"\n=== POBLACIÓN 0-4 LISTA ===")
print(f"Filas: {len(pob_filtrado)}")
print(f"Municipios: {pob_filtrado['cod_mun'].nunique()}")
print(f"Años: {sorted(pob_filtrado['año'].unique())}")
print(f"\nEjemplo:")
print(pob_filtrado.head(10))
print(f"\nEstadísticas pob_0_4:")
print(pob_filtrado['pob_0_4'].describe().round(0))


=== POBLACIÓN 0-4 LISTA ===
Filas: 7861
Municipios: 1123
Años: [2018, 2019, 2020, 2021, 2022, 2023, 2024]

Ejemplo:
  cod_mun   año  pob_0_4
0   05001  2018   142759
1   05002  2018     1430
2   05004  2018      224
3   05021  2018      454
4   05030  2018     2150
5   05031  2018     2299
6   05034  2018     3396
7   05036  2018      436
8   05038  2018     1038
9   05040  2018     2036

Estadísticas pob_0_4:
count      7861.0
mean       3384.0
std       16146.0
min           0.0
25%         497.0
50%        1121.0
75%        2533.0
max      478485.0
Name: pob_0_4, dtype: float64


**preparar la base del Índice de Pobreza Multidimensional (IPM)**

In [10]:
# Renombrar columnas
ipm.columns = [
    'cod_mun',
    'municipio',
    'analfabetismo',
    'bajo_logro_educativo',
    'barreras_primera_infancia',
    'barreras_acceso_salud',
    'tasa_dependencia',
    'hacinamiento_critico',
    'inadec_eliminacion_excretas',
    'inasistencia_escolar',
    'inadec_paredes',
    'inadec_pisos',
    'rezago_escolar',
    'sin_agua_mejorada',
    'sin_aseguramiento_salud',
    'trabajo_infantil',
    'trabajo_informal'
]

# Estandarizar código DIVIPOLA a string de 5 dígitos
ipm['cod_mun'] = ipm['cod_mun'].astype(str).str.zfill(5)

# Separar código departamento (primeros 2 dígitos)
ipm['cod_dpto'] = ipm['cod_mun'].str[:2]

# Eliminar filas que sean totales departamentales (tienen cod_mun terminado en 000)
ipm = ipm[~ipm['cod_mun'].str.endswith('000')].reset_index(drop=True)

print("=== IPM CENSAL 2018 ===")
print(f"Municipios: {len(ipm)}")
print(f"Variables: {len(ipm.columns) - 3}")  # sin cod_mun, municipio, cod_dpto
print(f"\nPrimeras filas:")
print(ipm[['cod_mun', 'municipio', 'analfabetismo', 'hacinamiento_critico', 'trabajo_infantil']].head())
print(f"\nEstadísticas básicas:")
print(ipm.drop(columns=['cod_mun', 'municipio', 'cod_dpto']).describe().round(1))

=== IPM CENSAL 2018 ===
Municipios: 1122
Variables: 15

Primeras filas:
  cod_mun   municipio  analfabetismo  hacinamiento_critico  trabajo_infantil
0   05001    MEDELLÍN            5.0                   5.4               0.5
1   05002   ABEJORRAL           18.9                   5.2               3.6
2   05004    ABRIAQUÍ           13.0                   5.4               1.2
3   05021  ALEJANDRÍA           15.7                   4.1               1.5
4   05030       AMAGÁ           15.4                   4.8               0.8

Estadísticas básicas:
       analfabetismo  bajo_logro_educativo  barreras_primera_infancia  \
count         1122.0                1122.0                     1122.0   
mean            17.4                  67.0                        2.6   
std              8.0                  12.7                        3.3   
min              2.0                  18.3                        0.4   
25%             11.9                  61.0                        1.3   
50%  

**preparar la base del sivigila**

In [11]:
#1. revisar como vienen los datos antes del unirlas
dfs_raw = {
    2016: sivigila_2016,
    2017: sivigila_2017,
    2018: sivigila_2018,
    2019: sivigila_2019,
    2020: sivigila_2020,
    2021: sivigila_2021,
    2022: sivigila_2022,
    2023: sivigila_2023,
    2024: sivigila_2024,
}

# Ver qué columnas tiene cada año (para diagnóstico)
for año, df in dfs_raw.items():
    print(f"{año}: {df.shape[1]} columnas — {df.shape[0]} filas")

# Columnas presentes en todos los años
cols_comunes = set.intersection(*[set(df.columns) for df in dfs_raw.values()])
print(f"\nColumnas comunes a todos los años: {len(cols_comunes)}")
print(sorted(cols_comunes))

2016: 69 columnas — 9714 filas
2017: 69 columnas — 10641 filas
2018: 74 columnas — 15386 filas
2019: 74 columnas — 17693 filas
2020: 74 columnas — 10744 filas
2021: 74 columnas — 15924 filas
2022: 74 columnas — 21337 filas
2023: 73 columnas — 23286 filas
2024: 72 columnas — 24221 filas

Columnas comunes a todos los años: 67
['AJUSTE', 'ANO', 'AREA', 'CBMTE', 'COD_ASE', 'COD_DPTO_N', 'COD_DPTO_O', 'COD_DPTO_R', 'COD_EVE', 'COD_MUN_N', 'COD_MUN_O', 'COD_MUN_R', 'COD_PAIS_O', 'COD_PAIS_R', 'COD_PRE', 'COD_SUB', 'CON_FIN', 'Departamento_Notificacion', 'Departamento_ocurrencia', 'Departamento_residencia', 'EDAD', 'Estado_final_de_caso', 'FECHA_NTO', 'FEC_AJU', 'FEC_ARC_XL', 'FEC_CON', 'FEC_DEF', 'FEC_HOS', 'FEC_NOT', 'FM_FUERZA', 'FM_GRADO', 'FM_UNIDAD', 'GP_CARCELA', 'GP_DESMOVI', 'GP_DESPLAZ', 'GP_DISCAPA', 'GP_GESTAN', 'GP_INDIGEN', 'GP_MAD_COM', 'GP_MIGRANT', 'GP_OTROS', 'GP_POBICFB', 'GP_PSIQUIA', 'GP_VIC_VIO', 'GRU_POB', 'INI_SIN', 'Municipio_notificacion', 'Municipio_ocurrencia', 'Mu

In [12]:
# @title
# 2. Concat con join='outer'
# Columnas que necesitamos para el análisis
# (todas existen en la mayoría de años — el join outer las rellena con NaN si falta)
COLS_NECESARIAS = [
    'COD_EVE',      # código evento
    'ANO',          # año de notificación
    'COD_DPTO_O',   # departamento de ocurrencia
    'COD_MUN_O',    # municipio de ocurrencia
    'EDAD',         # edad del caso
    'UNI_MED',      # unidad de medida de edad (años/meses/días)
    'SEXO',         # sexo
    'AREA',         # área (urbano/rural)
    'PER_ETN',      # pertenencia étnica
    'TIP_SS',       # tipo de seguridad social
    'PAC_HOS',      # fue hospitalizado
    'CON_FIN',      # condición final (vivo/muerto)
]


# Concat
sivigila_raw = pd.concat(
    dfs_raw.values(),
    ignore_index=True,
    join='outer'
)

print(f"\nTotal registros concat: {len(sivigila_raw)}")
print(f"Total columnas: {sivigila_raw.shape[1]}")

# Quedarse solo con columnas necesarias
cols_disponibles = [c for c in COLS_NECESARIAS if c in sivigila_raw.columns]
cols_faltantes   = [c for c in COLS_NECESARIAS if c not in sivigila_raw.columns]

if cols_faltantes:
    print(f"\n⚠️  Columnas no encontradas en ningún año: {cols_faltantes}")

sivigila = sivigila_raw[cols_disponibles].copy()
print(f"\nBase SIVIGILA consolidada: {sivigila.shape}")
print(sivigila.groupby('ANO').size().rename('casos'))


Total registros concat: 148946
Total columnas: 76

Base SIVIGILA consolidada: (148946, 12)
ANO
2016     9714
2017    10641
2018    15386
2019    17693
2020    10744
2021    15924
2022    21337
2023    23286
2024    24221
Name: casos, dtype: int64


In [13]:
# 3. Limpieza y estandarización
# Código DIVIPOLA de 5 dígitos
sivigila['cod_mun'] = (
    sivigila['COD_DPTO_O'].astype(str).str.zfill(2) +
    sivigila['COD_MUN_O'].astype(str).str.zfill(3)
)

# Año definitivo
sivigila['año'] = sivigila['ANO'].astype(int)

# Edad en meses (UNI_MED: 1=años, 2=meses, 3=días)
sivigila['edad_meses'] = np.where(
    sivigila['UNI_MED'] == 1, sivigila['EDAD'] * 12,
    np.where(sivigila['UNI_MED'] == 2, sivigila['EDAD'],
    np.where(sivigila['UNI_MED'] == 3, sivigila['EDAD'] / 30, np.nan))
)

# Banderas binarias
sivigila['fallecio']       = (sivigila['CON_FIN'] == 2).astype(int)
sivigila['hospitalizado']  = (sivigila['PAC_HOS'] == 1).astype(int)
sivigila['rural']          = (sivigila['AREA']    == 3).astype(int)
sivigila['indigena']       = (sivigila['PER_ETN'] == 1).astype(int)

print("=== SIVIGILA CONSOLIDADO ===")
print(f"Total casos: {len(sivigila)}")
print(f"Años cubiertos: {sorted(sivigila['año'].unique())}")
print(f"Municipios únicos: {sivigila['cod_mun'].nunique()}")
print(f"\nCasos por año:")
print(sivigila.groupby('año').agg(
    casos=('cod_mun', 'count'),
    fallecidos=('fallecio', 'sum'),
    hospitalizados=('hospitalizado', 'sum')
))

=== SIVIGILA CONSOLIDADO ===
Total casos: 148946
Años cubiertos: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Municipios únicos: 1160

Casos por año:
      casos  fallecidos  hospitalizados
año                                    
2016   9714           7            2407
2017  10641           0            2875
2018  15386           0            4144
2019  17693           0            4871
2020  10744           0            3232
2021  15924           0            3990
2022  21337           0            5820
2023  23286           0            7098
2024  24221           0            7026


In [14]:
sivigila.head()

,COD_EVE,ANO,COD_DPTO_O,COD_MUN_O,EDAD,UNI_MED,SEXO,AREA,PER_ETN,TIP_SS,PAC_HOS,CON_FIN,cod_mun,año,edad_meses,fallecio,hospitalizado,rural,indigena
0,113,2016,23,466,1,1,F,1,6,S,1,1,23466,2016,12.0,0,1,0,0
1,113,2016,54,1,1,1,M,1,6,N,2,1,54001,2016,12.0,0,0,0,0
2,113,2016,19,137,7,2,M,3,1,S,1,1,19137,2016,7.0,0,1,1,1
3,113,2016,54,1,2,1,F,1,6,P,2,1,54001,2016,24.0,0,0,0,0
4,113,2016,18,592,4,1,M,3,6,S,1,1,18592,2016,48.0,0,1,1,0


In [15]:
# 4. Agregar por municipio-año y merge con IPM

sivigila_mun = sivigila.groupby(['cod_mun', 'año']).agg(
    casos_totales     = ('fallecio',      'count'),
    fallecidos        = ('fallecio',      'sum'),
    hospitalizados    = ('hospitalizado', 'sum'),
    casos_rurales     = ('rural',         'sum'),
    casos_indigenas   = ('indigena',      'sum'),
    edad_media_meses  = ('edad_meses',    'mean')
).reset_index()

sivigila_mun['tasa_mortalidad']      = sivigila_mun['fallecidos']      / sivigila_mun['casos_totales']
sivigila_mun['tasa_hospitalizacion'] = sivigila_mun['hospitalizados']  / sivigila_mun['casos_totales']
sivigila_mun['prop_rural']           = sivigila_mun['casos_rurales']   / sivigila_mun['casos_totales']
sivigila_mun['prop_indigena']        = sivigila_mun['casos_indigenas'] / sivigila_mun['casos_totales']

print(f"Panel municipio-año: {len(sivigila_mun)} filas")
print(f"({sivigila_mun['cod_mun'].nunique()} municipios x {sivigila_mun['año'].nunique()} años)")

# Merge con IPM
base_panel = sivigila_mun.merge(ipm, on='cod_mun', how='left')

# Eliminar registros de exterior (cod_mun empieza en 00 o 01 con códigos inválidos)
base_panel = base_panel[base_panel['cod_mun'].str[:2].isin(
    [str(i).zfill(2) for i in range(5, 100)]  # departamentos válidos Colombia: 05-99
)].reset_index(drop=True)

print(f"\nBase panel final: {base_panel.shape}")
print(f"Nulos en IPM: {base_panel['analfabetismo'].isna().sum()}")

Panel municipio-año: 7964 filas
(1160 municipios x 9 años)

Base panel final: (7914, 29)
Nulos en IPM: 66


In [16]:
base_panel.head()

,cod_mun,año,casos_totales,fallecidos,hospitalizados,casos_rurales,casos_indigenas,edad_media_meses,tasa_mortalidad,tasa_hospitalizacion,...,inadec_eliminacion_excretas,inasistencia_escolar,inadec_paredes,inadec_pisos,rezago_escolar,sin_agua_mejorada,sin_aseguramiento_salud,trabajo_infantil,trabajo_informal,cod_dpto
0,05000,2016,2,0,1,2,0,7.000000,0.0,0.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,05000,2017,4,0,0,1,0,15.250000,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,05000,2018,5,0,1,1,1,14.200000,0.0,0.200000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,05000,2019,1,0,0,0,1,11.000000,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05001,2016,235,0,38,3,1,18.400284,0.0,0.161702,...,2.0,3.1,0.7,0.2,12.5,1.5,15.9,0.5,71.0,05


**Unión con los datos de población**

In [17]:
# 1. Merge con base panel IPM y la tasa de incidencia

base_panel = base_panel.merge(
    pob_filtrado,
    on=['cod_mun', 'año'],
    how='left'
)

# Tasa por cada 1.000 menores de 5 años
base_panel['tasa_x1000'] = (
    base_panel['casos_totales'] / base_panel['pob_0_4'] * 1000
).round(2)

print("=== TASA DE INCIDENCIA ===")
print(f"Nulos en pob_0_4: {base_panel['pob_0_4'].isna().sum()}")
print(f"Nulos en tasa:    {base_panel['tasa_x1000'].isna().sum()}")
print(f"\nDistribución:")
print(base_panel['tasa_x1000'].describe().round(2))

# Face validity
print("\nTop 15 municipios por tasa promedio:")
print(
    base_panel.groupby('cod_mun')['tasa_x1000']
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .round(2)
)

# Eliminar los datos que no tienen sentido
base_panel['tasa_x1000'] = base_panel['tasa_x1000'].replace([np.inf, -np.inf], np.nan)
base_panel = base_panel[base_panel['tasa_x1000'].notna()].copy()

print(f"✓ {len(base_panel):,} filas · {base_panel['cod_mun'].nunique():,} municipios · {base_panel['año'].nunique()} años")
print(f"✓ Rango tasa: [{base_panel['tasa_x1000'].min():.1f}, {base_panel['tasa_x1000'].max():.1f}]")

=== TASA DE INCIDENCIA ===
Nulos en pob_0_4: 1513
Nulos en tasa:    1513

Distribución:
count    6401.00
mean         inf
std          NaN
min         0.10
25%         2.16
50%         3.97
75%         6.88
max          inf
Name: tasa_x1000, dtype: float64

Top 15 municipios por tasa promedio:
cod_mun
94663      inf
97889    40.93
66572    40.84
99001    37.86
27073    37.41
50325    28.26
27025    26.34
44847    26.05
50450    24.06
68522    23.53
15022    22.93
27245    22.34
15621    22.31
15660    21.22
44001    21.20
Name: tasa_x1000, dtype: float64
✓ 6,399 filas · 1,109 municipios · 7 años
✓ Rango tasa: [0.1, 85.5]


**Unión de las coordenadas**

In [18]:
# Merge
base_panel = base_panel.merge(
    df_coord[['DIVIPOLA', 'x', 'y', 'NOMBRE', 'AREA']],
    left_on='cod_mun',
    right_on='DIVIPOLA',
    how='left'
)

base_panel = base_panel.rename(columns={'x': 'longitud', 'y': 'latitud'})

In [19]:
base_panel.head()

,cod_mun,año,casos_totales,fallecidos,hospitalizados,casos_rurales,casos_indigenas,edad_media_meses,tasa_mortalidad,tasa_hospitalizacion,...,trabajo_infantil,trabajo_informal,cod_dpto,pob_0_4,tasa_x1000,DIVIPOLA,longitud,latitud,NOMBRE,AREA
0,05001,2018,584,0,79,3,3,18.426142,0.0,0.135274,...,0.5,71.0,05,142759.0,4.09,05001,-75.611146,6.258455,Medellín,387.0
1,05001,2019,568,0,53,3,1,17.859683,0.0,0.093310,...,0.5,71.0,05,141423.0,4.02,05001,-75.611146,6.258455,Medellín,387.0
2,05001,2020,279,0,40,2,2,18.116846,0.0,0.143369,...,0.5,71.0,05,141008.0,1.98,05001,-75.611146,6.258455,Medellín,387.0
3,05001,2021,483,0,80,9,2,18.078123,0.0,0.165631,...,0.5,71.0,05,139148.0,3.47,05001,-75.611146,6.258455,Medellín,387.0
4,05001,2022,838,0,155,14,7,22.431702,0.0,0.184964,...,0.5,71.0,05,136515.0,6.14,05001,-75.611146,6.258455,Medellín,387.0


**Creación de la variable dependiente final**

In [20]:
# VARIABLE OBJETIVO FINAL

base_panel = base_panel.dropna(subset=['tasa_x1000']).copy()

base_panel['riesgo'] = base_panel.groupby('año')['tasa_x1000'].transform(
    lambda x: pd.qcut(x, q=3, labels=['bajo', 'medio', 'alto'], duplicates='drop')
)

print("=== VARIABLE OBJETIVO ===")
print(base_panel['riesgo'].value_counts().sort_index())
print(f"\nTasa promedio por categoría:")
print(base_panel.groupby('riesgo')['tasa_x1000'].agg(['mean','min','max']).round(2))
print(f"\nPor año:")
print(base_panel.groupby(['año','riesgo']).size().unstack(fill_value=0))

base_panel.to_csv('base_panel_final.csv', index=False)
print(f"\n✓ Base panel final: {base_panel.shape}")

=== VARIABLE OBJETIVO ===
riesgo
bajo     2136
medio    2132
alto     2133
Name: count, dtype: int64

Tasa promedio por categoría:
         mean   min    max
riesgo                    
bajo     1.69  0.10   3.68
medio    4.01  1.70   6.71
alto    10.99  4.06  85.55

Por año:
riesgo  bajo  medio  alto
año                      
2018     291    291   292
2019     302    301   302
2020     281    279   280
2021     300    302   299
2022     309    306   308
2023     324    323   324
2024     329    330   328

✓ Base panel final: (6401, 37)


**Base final con la variable objetivo y variables explicativas**

In [21]:
# 1. Definimos la lista exacta de columnas que se quedan
columnas_dane_y_target = [
    # Identificadores y Control
    'cod_mun',
    'año',
    'NOMBRE',

    # Variables Explicativas (15 dimensiones IPM - DANE)
    'analfabetismo',
    'bajo_logro_educativo',
    'barreras_primera_infancia',
    'barreras_acceso_salud',
    'tasa_dependencia',
    'hacinamiento_critico',
    'inadec_eliminacion_excretas',
    'inasistencia_escolar',
    'inadec_paredes',
    'inadec_pisos',
    'rezago_escolar',
    'sin_agua_mejorada',
    'sin_aseguramiento_salud',
    'trabajo_infantil',
    'trabajo_informal',
    'longitud',
    'latitud',
    'tasa_x1000',
    'casos_totales',
    'riesgo'      # Variable Objetivo
]




# 2. Creamos el nuevo DataFrame filtrado
df_modelo = base_panel[columnas_dane_y_target].copy()

#unimos con los indicadores

mdm_indicadores['Código Entidad'] = mdm_indicadores['Código Entidad'].str.zfill(5)

df_modelo = df_modelo.merge(
    mdm_indicadores[['Código Entidad', 'Año', 'Componente de gestión', 'Componente de resultados', 'MDM']],
    left_on=['cod_mun', 'año'],
    right_on=['Código Entidad', 'Año'],
    how='left'
)


# 3. Verificamos la forma final del dataset
print(f"Dataset filtrado listo. Forma: {df_modelo.shape}")

# 4. Guardamos la base de datos
df_modelo.to_csv(RUTA_DATOS_PROCESADOS / 'df_modelo_final.csv', index=False)

Dataset filtrado listo. Forma: (6401, 28)
